In [ ]:
import pdfplumber
import pandas as pd
import re

def extrair_pitstops_regex(caminho_pdf, caminho_csv_saida):
    print("Lendo o PDF como texto")
    dados_totais = []

    padrao = re.compile(r"^(\d+)\s+(.+?)\s+(\d+)\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s*([\d:\.]+)?$")

    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            for pagina in pdf.pages:
                texto = pagina.extract_text()
                if texto:
                    for linha in texto.split('\n'):
                        match = padrao.match(linha.strip())
                        if match and match.group(5):
                            dados_totais.append([match.group(1), match.group(2).strip(),
                                                 match.group(3), match.group(4), match.group(5)])
    except FileNotFoundError:
        print(f"ALERTA: O arquivo {caminho_pdf} não foi encontrado.")
        return

    if not dados_totais:
        print("Nenhum pit stop encontrado! O padrão do texto pode estar diferente.")
        return

    df_pitstops = pd.DataFrame(dados_totais,
                               columns=['Carro', 'Piloto_Equipe', 'Lap', 'Time of Day', 'Pit Time'])

    # --- Validação da extração ---
    paradas_por_carro = df_pitstops.groupby('Carro').size().sort_values(ascending=False)
    print(f"Extração: {len(df_pitstops)} pit stops de {df_pitstops['Carro'].nunique()} carros.")
    print(f"Paradas por carro — mín: {paradas_por_carro.min()} | máx: {paradas_por_carro.max()}")
    print(paradas_por_carro.to_string())

    df_pitstops.to_csv(caminho_csv_saida, index=False, sep=';', encoding='utf-8-sig')
    print(f"Salvo em: {caminho_csv_saida}")
    return df_pitstops

# --- ÁREA DE EXECUÇÃO ---
arquivo_pdf_pit = '../data/01_raw/pit_curvelo_p1.pdf'
arquivo_csv_pit = '../data/02_interim/dados_pitstops_P1.csv'

extrair_pitstops_regex(arquivo_pdf_pit, arquivo_csv_pit)

Lendo o PDF como texto
Pit Stops extraídos com perfeição! Salvo em: ../data/02_interim/dados_pitstops_P1.csv
